In [1]:
import os # Configure which GPU 
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e) 

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

resolution = [480,320] # increase for higher quality of renderings

# Define magic cell command to skip a cell if needed
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)



# Other imports          
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colormaps
import numpy as np
import sys
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera, watt_to_dbm

from sionna.phy.channel import OFDMChannel, CIRDataset
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.utils import ebnodb2no, PlotBER
from sionna.phy.ofdm import KBestDetector, LinearDetector
from sionna.phy.mimo import StreamManagement

# Import Sionna RT components
from sionna.rt import load_scene, Camera, Transmitter, Receiver, PlanarArray,\
                      PathSolver, RadioMapSolver
import random

In [2]:
scene = load_scene() # Load empty scene

# Configure antenna arrays for all transmitters and receivers
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,  # relative to wavelength
                             horizontal_spacing=0.5,  # relative to wavelength
                             pattern="iso",
                             polarization="V")
scene.rx_array = scene.tx_array

In [3]:
# Remove transmitters here so that the cell can be executed multiple times
scene.remove("tx0")
scene.remove("tx1")
scene.remove("tx2")
scene.remove("tx3")
scene.remove("tx4")

# Define and add a first transmitter to the scene
tx0 = Transmitter(name='tx0',
                  position=[-100, -100, 20],
                  orientation=[np.pi*5/6, 0, 0],
                  power_dbm=30)
scene.add(tx0)

tx1 = Transmitter(name='tx1',
                  position=[-100, 50, 20],
                  orientation=[np.pi/6, 0, 0],
                  power_dbm=30)
scene.add(tx1)

tx2 = Transmitter(name='tx2',
                  position=[100, -100, 20],
                  orientation=[-np.pi/2, 0, 0],
                  power_dbm=30)
scene.add(tx2)

rm_solver = RadioMapSolver()
tx3 = Transmitter(name='tx3',
                  position=[-50, -50, 20],
                  orientation=[np.pi/6, 0, 0],
                  power_dbm=30)
scene.add(tx3)

tx4 = Transmitter(name='tx4',
                  position=[50, 100, 20],
                  orientation=[-np.pi/2, 0, 0],
                  power_dbm=30)
scene.add(tx4)



rm = rm_solver(scene,
               max_depth=5,
               cell_size=(1., 1.),
               samples_per_tx=10**7)
scene.preview(  radio_map=rm)

Renderer(camera=PerspectiveCamera(aspect=1.31, children=(DirectionalLight(intensity=0.25, position=(0.0, 0.0, …

In [4]:
# 獲取單元格中心的 x 和 y 坐標
x_unique = cm.cell_centers[0, :, 0]
y_unique = cm.cell_centers[:, 0, 1]

# sinr_0_db = 10 * np.log10(cm.sinr[0])
# plt.pcolormesh(x_unique, y_unique, sinr_0_db, vmin=-25, vmax=20, shading='auto')
# plt.colorbar(label='SINR (dB)')
# plt.title('SINR for tx0')
# plt.xlabel('x (m)')
# plt.ylabel('y (m)')
# plt.show()

# 提取 SINR 並轉換為 NumPy 陣列
sinr_0 = cm.sinr[0].numpy()  # tx0 的 SINR
sinr_1 = cm.sinr[1].numpy()  # tx1 的 SINR
sinr_2 = cm.sinr[2].numpy()  # tx2 的 SINR

# 將陣列放入一個 list
sinr_list = [sinr_0, sinr_1, sinr_2]

# 使用 np.maximum.reduce() 計算逐點最大值
sinr_max_01 = np.maximum.reduce(sinr_list)

# 轉換為 dB
sinr_max_01_db = 10 * np.log10(sinr_max_01)

# 繪製覆蓋圖
plt.pcolormesh(x_unique, y_unique, sinr_max_01_db, vmin=-25, vmax=20, shading='auto')
plt.colorbar(label='SINR (dB)')
plt.xlabel('x (m)')
plt.ylabel('y (m)')
# plt.title('Maximum SINR between tx0 and tx1 with tx2 as interference')

# 標記發射機位置
tx0_pos = scene.transmitters['tx0'].position
tx1_pos = scene.transmitters['tx1'].position
tx2_pos = scene.transmitters['tx2'].position
tx3_pos = scene.transmitters['tx3'].position
tx4_pos = scene.transmitters['tx4'].position

plt.scatter(tx0_pos[0], tx0_pos[1], c='red', label='tx0', marker='o', s=10)  # 紅色圓點表示 tx0
plt.scatter(tx1_pos[0], tx1_pos[1], c='red', label='tx1', marker='o', s=10)  # 綠色方形表示 tx1
plt.scatter(tx2_pos[0], tx2_pos[1], c='red', label='tx2', marker='o', s=10)  # 藍色三角形表示 tx2
plt.scatter(tx3_pos[0], tx3_pos[1], c='black', label='tx3', marker='^', s=10)  # 藍色三角形表示 tx2
plt.scatter(tx4_pos[0], tx4_pos[1], c='black', label='tx4', marker='^', s=10)  # 藍色三角形表示 tx2
plt.legend()

plt.show()

NameError: name 'cm' is not defined